# Visualization and Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/05_visualization_and_analysis.ipynb)

`catsim.plot` contains the high-level plots that most users need when they
are exploring an item bank or interpreting a CAT simulation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from catsim import plot
from catsim.engine import CatEngine
from catsim.estimation import NumericalSearchEstimator
from catsim.initialization import UniformInitializer
from catsim.item_bank import ItemBank
from catsim.selection import MaxInfoSelector
from catsim.simulation import SimulationRunner
from catsim.stopping import MaxLengthStopper

In [ ]:
item_bank = ItemBank.generate(120, rng=41)
engine = CatEngine(
  item_bank=item_bank,
  initializer=UniformInitializer(),
  selector=MaxInfoSelector(),
  estimator=NumericalSearchEstimator(),
  stopper=MaxLengthStopper(12),
)
true_thetas = np.random.default_rng(3).normal(0, 1, 25)
result = SimulationRunner(engine).run(true_thetas, rng=101)

## Plot one item

In [ ]:
item = item_bank.items[5]
fig, axes = plt.subplots(3, 1, figsize=(7, 12))
plot.item_curve(item[0], item[1], item[2], item[3], ptype=plot.PlotType.ICC, ax=axes[0])
plot.item_curve(item[0], item[1], item[2], item[3], ptype=plot.PlotType.IIC, ax=axes[1])
plot.item_curve(item[0], item[1], item[2], item[3], ptype=plot.PlotType.BOTH, ax=axes[2])
plt.tight_layout()
plt.show()
plt.close("all")

## Visualize the item bank

Each point is one item, placed by its discrimination `a`, difficulty `b` and
guessing `c`: the spread along `b` shows the ability range the bank covers,
and the items with high `a` are the ones maximum-information selection
administers most often.

In [ ]:
plot.gen3d_dataset_scatter(item_bank, title="Item Bank Geometry")
plt.show()
plt.close("all")

## Visualize one session trace

The lines share one y axis but not one unit: $\hat{\theta}$, the true $\theta$
(dashed) and the difficulty of each administered item are on the ability
scale, $I(\theta)$ is the test information, $Var$ and $SEE$ are the variance
and standard error of $\hat{\theta}$ ($Var = 1/I(\theta)$, $SEE = \sqrt{Var}$),
and reliability is $1 - 1/I(\theta)$, which is negative while the test
information is below 1, as it is over the first few items.

In [ ]:
plot.test_progress(simulation=result, index=0, info=True, var=True, see=True, reliability=True, figsize=(9, 5))
plt.tight_layout()
plt.show()
plt.close("all")

## Visualize exposure patterns

The upper panel shows each item's exposure rate in item-bank order, which has
no meaning of its own, and the histogram below counts items by exposure rate:
most items are never administered, while a few go to most examinees.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 8))
plot.item_exposure(simulation=result, ax=axes[0], title="Exposure by Item Order")
plot.item_exposure(simulation=result, ax=axes[1], title="Exposure Histogram", hist=True)
plt.tight_layout()
plt.show()
plt.close("all")

## Reporting guidance

- Item curves help explain the meaning of item parameters.
- Session progress plots show how the CAT converged for one examinee.
- Exposure plots help evaluate item security and distribution.

These figures are descriptive. They are useful for communicating results,
but they should be interpreted together with design choices such as test
length, stopper settings, and item-bank calibration quality.